# Microsoft Agent Framework — Azure OpenAI (Responses API)

У овом примеру кода користићете **Microsoft Agent Framework (MAF)** за креирање једноставног агента подржаног од стране **Azure OpenAI** користећи **Responses API**.

> **Белешка о миграцији:** Овај пример је раније користио Semantic Kernel са GitHub Models. Сада је мигриран на Microsoft Agent Framework, а GitHub Models (застарело, престаје са радом у јулу 2026) је замењен Azure OpenAI који подржава Responses API. `OpenAIChatClient` у MAF-у циља стабилан Azure OpenAI `/openai/v1/` крајњи приступ и подразумевано користи Responses API.

Сврха овог примера је да демонстрира кораке који ће касније бити примењени у додатним примерима кода приликом имплементације различитих агенцијских образаца.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Увези потребне Python пакете


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Дефинисање алата

У Microsoft Agent Framework-у, **алат** је обична Python функција украшена са `@tool` коју агент може позвати. Испод дефинишемо алат који враћа случајну дестинацију за одмор и избегава понављање претходне.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Креирање агента

Овде креирамо агента по имену `TravelAgent`.

У овом примеру користимо врло једноставна упутства. Слободно измените ова упутства да бисте видели како се понашање агента мења.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Покретање агента

Сада можемо покренути агента. Креирамо `AgentSession` како би агент памтио разговор током узастопних интеракција, затим шаљемо два `user_inputs`. Први тражи путовање; други каже да кориснику није био по вољи предлог и тражи други — агент користи историју сесије плус алат `get_random_destination` да одговори.

Можете изменити ове поруке да бисте видели како агент другачије реагује. Одговори се **стрим-ују** токен по токен.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
